# Notebook 03 — Statistical Validation

Clustering pace profiles and running the statistical tests that validate (or reject) the patterns from the feature engineering step. `race.parquet` gets merged in here for track condition and distance.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss

PROCESSED_DATA_DIR = Path("../data/processed")

features = pd.read_parquet(PROCESSED_DATA_DIR / "horse_race_features.parquet")
race = pd.read_parquet(PROCESSED_DATA_DIR / "race.parquet")

print("features:", features.shape)
print("race:", race.shape)

features: (14914, 17)
race: (2000, 10)


In [2]:
features = features.merge(
    race[["track_id", "race_date", "race_number", "track_condition", "distance_id"]],
    on=["track_id", "race_date", "race_number"],
    how="left",
)
features[["track_condition", "distance_id"]].isnull().sum()

track_condition    0
distance_id        0
dtype: int64

## 1. Pace profile clustering

K-means on the three segment speeds (early/mid/late), standardized first since k-means is distance-based. Using k=3 because racing naturally breaks into three strategy types (front-runner, presser, closer), rather than searching for the optimal k.

In [3]:
X = features[["speed_early", "speed_mid", "speed_late"]]
X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
features["pace_cluster"] = kmeans.fit_predict(X_scaled)

features.groupby("pace_cluster")[["speed_early", "speed_mid", "speed_late"]].mean()

,speed_early,speed_mid,speed_late
pace_cluster,,,
0,18.915101,18.597533,15.559251
1,18.042695,19.151129,18.386777
2,19.370773,19.938515,17.750437


Three distinct shapes emerge:
- **Cluster 0**: fast early (18.9), fades hard by the end (15.6, a drop of ~3 m/s). Goes out fast, can't sustain it.
- **Cluster 1**: slowest start (18.0), but holds pace best late (18.4, barely any drop). The "stamina" profile.
- **Cluster 2**: fastest across the board (19.4 early, 19.9 mid, 17.8 late). Still fades some, but its overall speed is high enough that it doesn't matter as much.

Naming them descriptively rather than assuming the classic "front-runner / presser / closer" labels apply exactly, these are based on speed shape, not on actual running position (which we already have separately as `position_change`).

In [4]:
features["pace_cluster"].value_counts()

pace_cluster
2    6076
0    4777
1    4061
Name: count, dtype: int64

## 2. Does pace profile predict the result?

In [5]:
features.groupby("pace_cluster")["position_at_finish"].mean()

pace_cluster
0    5.132091
1    4.427235
2    4.017281
Name: position_at_finish, dtype: float64

Cluster 2 (sustained high speed) finishes best on average (4.0), Cluster 1 (stamina/closer type) is in the middle (4.4), and Cluster 0 (fast-start faders) finishes worst (5.1). Testing whether this difference is real or could be chance.

In [6]:
groups = [g["position_at_finish"].values for _, g in features.groupby("pace_cluster")]
stat, p_value = stats.kruskal(*groups)
print(f"H = {stat:.1f}, p = {p_value:.2e}")

H = 514.5, p = 1.88e-112


p-value is essentially zero. The difference between pace profiles is real, not something you'd see by chance. Going out fast and fading is the worst strategy of the three; sustained overall speed wins out.

## 3. What explains fatigue?

In [7]:
model = smf.ols(
    "fatigue_z ~ C(track_condition, Treatment(reference='FT')) + distance_id + weight_carried",
    data=features,
).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              fatigue_z   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                  0.004617
Date:                Sun, 21 Jun 2026   Prob (F-statistic):               1.00
Time:                        20:42:10   Log-Likelihood:                -20088.
No. Observations:               14914   AIC:                         4.019e+04
Df Residuals:                   14905   BIC:                         4.026e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                                                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------

R² of 0.000. Nothing explains anything, every p-value near 1. That's suspicious enough to investigate rather than just report as a null result.

In [8]:
features.groupby("track_condition")["fatigue_z"].agg(["mean", "std"])

,mean,std
track_condition,,
FM,6.628197e-18,0.938664
FT,6.786479e-18,0.926712
GD,9.874711e-18,0.936005
MY,1.184018e-16,0.924592
SF,1.519253e-16,0.942809
SY,3.292358e-17,0.922929
YL,2.757136e-17,0.933333


The mean of `fatigue_z` is essentially zero (to many decimal places) in every single track condition. That's not a coincidence, it's a direct consequence of how we built it in notebook 02: `fatigue_z` is standardized *within each race*, which by construction removes all race-level variance. Since `track_condition` and `distance_id` are constant within a race, there's nothing left for them to explain in the z-scored version. This isn't a real null finding, it's a methodology mismatch. Using `fatigue_raw` (before standardization) instead, since we need cross-race variation preserved to test race-level predictors.

In [9]:
model = smf.ols(
    "fatigue_raw ~ C(track_condition, Treatment(reference='FT')) + distance_id + weight_carried",
    data=features,
).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            fatigue_raw   R-squared:                       0.338
Model:                            OLS   Adj. R-squared:                  0.338
Method:                 Least Squares   F-statistic:                     952.9
Date:                Sun, 21 Jun 2026   Prob (F-statistic):               0.00
Time:                        20:42:10   Log-Likelihood:                -22932.
No. Observations:               14914   AIC:                         4.588e+04
Df Residuals:                   14905   BIC:                         4.595e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                                                          coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------

R² = 0.34 and most predictors significant at p < 0.001, a very different picture from the `fatigue_z` model.

- **`weight_carried`**: positive. Heavier weight is associated with more fatigue, which makes sense biomechanically.
- **`distance_id`**: negative. Longer races show less of a mid-to-late speed drop, horses seem to pace themselves more evenly, while sprints produce a sharper fade.
- **`track_condition`**: SY (sloppy) shows more fatigue than FT (fast/dry). Running through wet footing is more taxing. Most other conditions show less fatigue than FT, possibly because fast tracks push horses harder and raise the baseline effort.

One known limitation: horses within the same race share track condition and distance, so they're not fully independent observations. Clustered standard errors by race would be more rigorous, worth flagging, not implementing here.

## 4. Market calibration

Does the market's implied probability actually predict winning? Logistic regression of `won` on `implied_prob`.

In [10]:
features["won"] = (features["position_at_finish"] == 1).astype(int)
model_data = features.dropna(subset=["implied_prob"])

model = smf.logit("won ~ implied_prob", data=model_data).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.339865
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                    won   No. Observations:                14911
Model:                          Logit   Df Residuals:                    14909
Method:                           MLE   Df Model:                            1
Date:                Sun, 21 Jun 2026   Pseudo R-squ.:                  0.1380
Time:                        20:42:10   Log-Likelihood:                -5067.7
converged:                       True   LL-Null:                       -5879.2
Covariance Type:            nonrobust   LLR p-value:                     0.000
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -3.1092      0.046    -67.621      0.000      -3.199      -3.019
implied_prob     6.0006

In [11]:
pred = model.predict(model_data)
auc = roc_auc_score(model_data["won"], pred)
print(f"AUC: {auc:.3f}")

AUC: 0.778


Coefficient is positive and highly significant (p < 0.001), and AUC of 0.78 means the market's own pricing does a solid job telling winners from non-winners. Well above the 0.5 you'd get from random guessing.

In [12]:
brier_model = brier_score_loss(model_data["won"], pred)

base_rate = model_data["won"].mean()
brier_baseline = brier_score_loss(model_data["won"], np.full(len(model_data), base_rate))

print(f"Brier score using implied_prob: {brier_model:.4f}")
print(f"Brier score, no-skill baseline (always predict the average): {brier_baseline:.4f}")

Brier score using implied_prob: 0.1021
Brier score, no-skill baseline (always predict the average): 0.1162


0.102 vs. 0.116. Using the market's odds beats predicting the average win rate for every horse. Combined with the AUC of 0.78, the market's relative pricing is genuinely informative, even if the absolute implied probabilities run a bit high due to the track's takeout.

Giving the clusters descriptive names so the Power BI visuals are readable. Using each cluster's mean fatigue to assign labels dynamically, so the mapping stays correct if the cluster numbers shift on a re-run.

In [13]:
cluster_fatigue = features.groupby("pace_cluster")["fatigue_raw"].mean().sort_values()
ordered_clusters = cluster_fatigue.index.tolist()

label_map = {
    ordered_clusters[0]: "Stamina / closer",
    ordered_clusters[1]: "Sustained speed",
    ordered_clusters[2]: "Fast-start fader",
}
features["pace_cluster_label"] = features["pace_cluster"].map(label_map)

features["won"] = (features["position_at_finish"] == 1).astype(int)
features["prob_decile"] = pd.qcut(features["implied_prob"], 10, labels=False, duplicates="drop")

Adding `won` and `prob_decile` here too, both are trivial in pandas, and precomputing them means Power BI doesn't need to replicate this logic in DAX later.

## 5. Export final feature table (with cluster labels)

In [14]:
features.to_parquet(PROCESSED_DATA_DIR / "horse_race_features_final.parquet", index=False)

## Summary of statistical findings

- **Pacing strategy**: three profiles from clustering (fast-start fader, stamina/closer, sustained high speed). Finishing position differs significantly across them (Kruskal-Wallis, p < 0.001). Going out fast and fading is the weakest approach.
- **Fatigue**: weight, distance, and track condition together explain about a third of the variation in raw fatigue (R² = 0.34). Heavier weight and sloppy conditions are both associated with more fatigue; longer races less so.
- **Market calibration**: implied probability predicts winning with AUC = 0.78. The market ranks horses well, even if the absolute probabilities are inflated by the takeout.
- **Track bias (post position)**: not testable here, no starting-gate position field in the data. Track condition and distance effects (see fatigue section above) are the closest available proxies.